In [1]:
import pandas as pd
import numpy as np

In [2]:
country_vehicle = pd.read_excel('./Data Extraction Sheet.xlsx', sheet_name="Country-Vehicle Extraction")
country_vehicle = country_vehicle[country_vehicle.Country == 'India']
country_vehicle

/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Country,Vehicle,Quintile,Data need,Data point name,Units,Year,Data source,Value,CI,SE,Notes
84,India,Rice,All,"Vehicle ""fortifiability"" (essentially amount i...",percentage,%,2024.0,M4N Database,0.513,NaN,NaN,"""Industry consolidation"" sent by Jonathan Gors..."


In [3]:
assert (country_vehicle.Vehicle == 'Rice').all()

In [4]:
country_vehicle['Data need'].unique()

array(['Vehicle "fortifiability" (essentially amount industrially produced)'],
      dtype=object)

In [5]:
percent_data_needs = {
    'Vehicle "fortifiability" (essentially amount industrially produced)': "vehicle_fortifiability",
}

In [6]:
pregnancy_sim_data_dir = '../../0200_pregnancy_sim/src/vivarium_gates_lsff_by_wealth_quintile/data/raw_data/'
import pathlib
pathlib.Path(pregnancy_sim_data_dir).mkdir(parents=True, exist_ok=True)

In [7]:
def scale_to_total(rows):
    assert rows.Quintile.is_unique
    rows = rows.set_index('Quintile').Value
    total = rows.loc['All']
    print(f'Total: {total}')
    by_quintile = rows[rows.index != 'All']
    print('By quintile, before scaling')
    print(by_quintile)
    print(f'Implied total with equal weight: {by_quintile.mean()}')
    scale_factor = total / by_quintile.mean()
    by_quintile = (by_quintile * scale_factor).rename("value").reset_index().rename(columns={"Quintile": "wealth_quintile"})
    by_quintile["wealth_quintile"] = by_quintile.wealth_quintile.str.lower()
    print('By quintile, after scaling')
    print(by_quintile)
    return by_quintile

In [8]:
# TODO: HCES data should be informing the disparity
rows = pd.concat([
    country_vehicle,
    country_vehicle.assign(Quintile='Lowest'),
    country_vehicle.assign(Quintile='Second'),
    country_vehicle.assign(Quintile='Middle'),
    country_vehicle.assign(Quintile='Fourth'),
    country_vehicle.assign(Quintile='Highest'),
], ignore_index=True)

In [9]:
assert (rows.Units == '%').all()
assert (rows['Data point name'] == 'percentage').all()

result = scale_to_total(rows)
result.value = result.value.clip(0, 1)
result.insert(0, "vehicle_name", "rice")
result.to_csv(f'{pregnancy_sim_data_dir}/vehicle_fortifiability/india.csv', index=False)

Total: 0.513
By quintile, before scaling
Quintile
Lowest     0.513
Second     0.513
Middle     0.513
Fourth     0.513
Highest    0.513
Name: Value, dtype: float64
Implied total with equal weight: 0.513
By quintile, after scaling
  wealth_quintile  value
0          lowest  0.513
1          second  0.513
2          middle  0.513
3          fourth  0.513
4         highest  0.513


In [10]:
country_vehicle_fort = pd.read_excel('./Data Extraction Sheet.xlsx', sheet_name="Country-Vehicle-Fort Extraction")
country_vehicle_fort = country_vehicle_fort[(country_vehicle_fort.Country == 'India') & (country_vehicle_fort.Fortificant == 'Iron')]
country_vehicle_fort

/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Country,Vehicle,Fortificant,Quintile,Data need,Data point name,Units,Year,Data source,Value,CI,Notes
36,India,Rice,Iron,All,Vehicle fortification at baseline -- amount am...,NaN,mcg/g,NaN,"slide deck JG shared to ADF, credited to \nHel...",42.5,NaN,Originally in mg/100g; https://www.wolframalph...
37,India,Rice,Iron,Lowest,Vehicle fortification at baseline -- amount am...,NaN,mcg/g,NaN,Assumption,42.5,NaN,assumed same by quintile?
38,India,Rice,Iron,Second,Vehicle fortification at baseline -- amount am...,NaN,mcg/g,NaN,Assumption,42.5,NaN,NaN
39,India,Rice,Iron,Middle,Vehicle fortification at baseline -- amount am...,NaN,mcg/g,NaN,Assumption,42.5,NaN,NaN
40,India,Rice,Iron,Fourth,Vehicle fortification at baseline -- amount am...,NaN,mcg/g,NaN,Assumption,42.5,NaN,NaN
41,India,Rice,Iron,Highest,Vehicle fortification at baseline -- amount am...,NaN,mcg/g,NaN,Assumption,42.5,NaN,NaN


In [11]:
def check_with_total(rows):
    assert rows.Quintile.is_unique
    rows = rows.set_index('Quintile').Value
    total = rows.loc['All']
    print(f'Total: {total}')
    by_quintile = rows[rows.index != 'All']
    print('By quintile')
    print(by_quintile)
    assert np.isclose(by_quintile.mean(), total, atol=0, rtol=0.1)
    print(f'Implied total with equal weight: {by_quintile.mean()}')
    by_quintile = by_quintile.rename("value").reset_index().rename(columns={"Quintile": "wealth_quintile"})
    by_quintile["wealth_quintile"] = by_quintile.wealth_quintile.str.lower()
    return by_quintile

In [12]:
concentration_data_needs = {
    "Vehicle fortification at baseline -- amount among fortified": "baseline_iron_fortification_concentration",
}

for data_need_name, dir_name in concentration_data_needs.items():
    print(f'Data need: {data_need_name}')
    rows = country_vehicle_fort[country_vehicle_fort['Data need'] == data_need_name]
    assert (rows.Units == 'mcg/g').all()

    result = check_with_total(rows)
    result.insert(0, "vehicle_name", "rice")
    result.to_csv(f'{pregnancy_sim_data_dir}/{dir_name}/india.csv', index=False)

Data need: Vehicle fortification at baseline -- amount among fortified
Total: 42.5
By quintile
Quintile
Lowest     42.5
Second     42.5
Middle     42.5
Fourth     42.5
Highest    42.5
Name: Value, dtype: float64
Implied total with equal weight: 42.5


In [13]:
scenarios = pd.read_excel('./Data Extraction Sheet.xlsx', sheet_name="Scenario Definition Extraction")
scenarios = scenarios[(scenarios.Country == 'India') & (scenarios.Fortificant == 'Iron')]
assert (scenarios.Vehicle == 'Rice').all()
scenarios

/mnt/share/homes/zmbc/src/vivarium_gates_lsff_by_wealth_quintile/0100_data_prep/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Country,Vehicle,Fortificant,Scenario,Data need,Data point name,Units,Data source,Value,Notes
4,India,Rice,Iron,Intervention,Intervention coverage % of fortifiable and unf...,percentage,%,Andy Garcia email 7/26,0.8,Same across all vehicles/countries/fortificants
10,India,Rice,Iron,Intervention,Intervention effective % of newly fortified,percentage,%,Andy Garcia email 7/26,0.8,Same across all vehicles/countries/fortificants
16,India,Rice,Iron,Intervention,Vehicle fortification in intervention -- amoun...,NaN,mcg/g,FSSAI,42.5,"Same as baseline, PDS rice; see county-vehicle..."


In [14]:
percent_data_needs = {
    'Intervention coverage % of fortifiable and unfortified': "intervention_iron_fortification_coverage",
    'Intervention effective % of newly fortified': "intervention_iron_fortification_effective_coverage",
}

for data_need_name, dir_name in percent_data_needs.items():
    print(f'Data need: {data_need_name}')
    rows = scenarios[scenarios['Data need'] == data_need_name]
    assert (rows.Units == '%').all()
    assert (rows['Data point name'] == 'percentage').all()
    assert (rows['Scenario'] == 'Intervention').all()

    result = rows[["Vehicle", "Value"]].rename(columns={"Vehicle": "vehicle_name", "Value": "value"})
    result.to_csv(f'{pregnancy_sim_data_dir}/{dir_name}/india.csv', index=False)

Data need: Intervention coverage % of fortifiable and unfortified


Data need: Intervention effective % of newly fortified


In [15]:
concentration_data_needs = {
    "Vehicle fortification in intervention -- amount among fortified": "intervention_iron_fortification_concentration",
}

for data_need_name, dir_name in concentration_data_needs.items():
    print(f'Data need: {data_need_name}')
    rows = country_vehicle_fort[country_vehicle_fort['Data need'] == data_need_name]
    assert (rows.Units == 'mcg/g').all()

    result = rows[["Vehicle", "Value"]].rename(columns={"Vehicle": "vehicle_name", "Value": "value"})
    result.to_csv(f'{pregnancy_sim_data_dir}/{dir_name}/india.csv', index=False)

Data need: Vehicle fortification in intervention -- amount among fortified
